# 📦 Smart Inventory Intelligence — EDA & Modeling

This notebook walks through the full analytical pipeline:
1. Data loading & cleaning
2. Exploratory Data Analysis
3. Demand Forecasting with Prophet
4. Stockout Risk Classification
5. Sentiment Analysis with VADER

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src')

from data_pipeline import load_olist_data, build_master_df, build_category_weekly, build_sku_features, get_top_categories
from forecaster import forecast_category, get_forecast_summary
from risk_classifier import train_risk_model, predict_risk, get_risk_summary
from sentiment import score_reviews, merge_sentiment_with_orders, get_category_sentiment

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Data Loading

In [ ]:
orders, items, products, reviews, categories = load_olist_data()
master = build_master_df(orders, items, products, categories)
print(f'Orders: {len(orders):,} | Items: {len(items):,} | Products: {len(products):,}')
master.head()

## 2. Exploratory Data Analysis

In [ ]:
# Revenue over time
master['month'] = master['order_purchase_timestamp'].dt.to_period('M').dt.start_time
monthly_rev = master.groupby('month')['price'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
monthly_rev.plot(ax=axes[0], color='steelblue', linewidth=2.5)
axes[0].set_title('Monthly Revenue (R$)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('')

# Top categories
top_cats = master.groupby('product_category_name_english')['price'].sum().nlargest(10)
top_cats.sort_values().plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Top 10 Categories by Revenue', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Demand Forecasting

In [ ]:
weekly = build_category_weekly(master)
category = 'bed_bath_table'
forecast, history = forecast_category(weekly, category, periods=8)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(history['ds'], history['y'], color='steelblue', label='Actual', linewidth=2)
future = forecast[forecast['ds'] > history['ds'].max()]
ax.plot(future['ds'], future['yhat'], color='darkorange', label='Forecast', linewidth=2.5)
ax.fill_between(future['ds'], future['yhat_lower'], future['yhat_upper'], alpha=0.2, color='darkorange')
ax.set_title(f'Demand Forecast — {category}', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

get_forecast_summary(forecast, 8)

## 4. Stockout Risk Classification

In [ ]:
sku_df = build_sku_features(master)
model, le, feature_cols, report = train_risk_model(sku_df)
sku_pred = predict_risk(model, le, feature_cols, sku_df)

print('Classification Report:')
print(pd.DataFrame(report).T.round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sku_pred['predicted_risk'].value_counts().plot(kind='bar', ax=axes[0],
    color=['#ef4444','#10b981','#f59e0b'], edgecolor='white')
axes[0].set_title('SKU Risk Distribution', fontsize=13, fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

axes[1].scatter(sku_pred['velocity'], sku_pred['days_of_stock_est'],
    c=sku_pred['predicted_risk'].map({'High':'#ef4444','Medium':'#f59e0b','Low':'#10b981'}),
    alpha=0.4, s=15)
axes[1].set_xlabel('Weekly Velocity')
axes[1].set_ylabel('Days of Stock')
axes[1].set_title('Velocity vs Days-of-Stock', fontsize=13, fontweight='bold')
axes[1].set_ylim(0, 100)
plt.tight_layout()
plt.show()

## 5. Sentiment Analysis

In [ ]:
scored_reviews = score_reviews(reviews)
sent_merged = merge_sentiment_with_orders(scored_reviews, orders, items, products, categories)
cat_sentiment = get_category_sentiment(sent_merged)

fig, ax = plt.subplots(figsize=(10, 6))
worst10 = cat_sentiment.head(10)
colors = ['#ef4444' if s < 0 else '#f59e0b' if s < 0.2 else '#10b981' for s in worst10['avg_sentiment']]
ax.barh(worst10['category'], worst10['avg_sentiment'], color=colors)
ax.set_title('Avg Sentiment Score — Bottom 10 Categories', fontsize=14, fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()

cat_sentiment.head(15)

## 6. Key Business Insights

- Categories with high stockout risk AND declining sentiment need **immediate restocking + quality review**
- Demand spikes in forecasts that correlate with sentiment dips may indicate **viral complaints or returns surges**
- SKUs with high velocity but only 7 days of stock are **highest priority reorder targets**